In [ ]:
!pip install -q pandas scikit-learn tqdm

In [ ]:
import os
import json
import random

import pandas as pd

from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# Config
SEMANTIC_JSON = "/content/drive/MyDrive/5.SemanticMapping/semantic_output/semantic_blocks.json"

OUTPUT_DIR = "/content/drive/MyDrive/5.SemanticMapping/scope_dataset"

os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(42)

print("Ready.")

Ready.


In [ ]:
# Load semantic blocks
with open(SEMANTIC_JSON, "r", encoding="utf-8") as f:
    semantic_blocks = json.load(f)

print("Total semantic blocks:", len(semantic_blocks))

Total semantic blocks: 13236


In [ ]:
# Inspect structure
print(json.dumps(semantic_blocks[0], indent=2, ensure_ascii=False))

{
  "block_id": 0,
  "page": "bao_viet_holdings_2023_p002_jpg.rf.6d94b20408c83a64a3f18835cb476455.jpg",
  "bbox": [
    851,
    379,
    960,
    424
  ],
  "type": "figure",
  "confidence": 0.560416255146265,
  "text": "Ggo0 APPRECIATE WHAT WE HAVE\nENKIRONMENTAL what we hold",
  "top_matches": [
    {
      "chunk_id": 2015,
      "document": "GRI 2_ General Disclosures 2021",
      "standard": "GRI 2",
      "page": 52,
      "local_chunk": 2,
      "num_words": 20,
      "text": "; modified Note: See Guidance to Disclosure 2-25 in GRI 2: General Disclosures 2021 for more information on ‘grievance mechanism’",
      "score": 0.839975893497467
    },
    {
      "chunk_id": 998,
      "document": "GRI 13_ Agriculture Aquaculture and Fishing Sectors 2022 V1.1",
      "standard": "GRI 13",
      "page": 24,
      "local_chunk": 2,
      "num_words": 21,
      "text": ". This information can support the reporting for additional sector disclosure 13.4.5. GRI 13: Agriculture, Aquaculture

In [ ]:
rows = []

for item in semantic_blocks:

    rows.append({

        "block_id": item["block_id"],

        "page": item["page"],

        "text": item["text"],

        "type": item["type"],

        "confidence": item["confidence"],

        "semantic_standard": item["semantic_standard"],

        "semantic_document": item["semantic_document"],

        "semantic_score": item["semantic_score"],

        "scope": ""

    })

df = pd.DataFrame(rows)

print(df.shape)

df.head()

(13236, 9)


,block_id,page,text,type,confidence,semantic_standard,semantic_document,semantic_score,scope
0,0,bao_viet_holdings_2023_p002_jpg.rf.6d94b20408c...,Ggo0 APPRECIATE WHAT WE HAVE\nENKIRONMENTAL wh...,figure,0.560416,GRI 2,GRI 2_ General Disclosures 2021,0.839976,
1,1,bao_viet_holdings_2023_p002_jpg.rf.6d94b20408c...,60 9400\nSOCIAL Ve'll be shoring,figure,0.446182,GRI 12,GRI 12_ Coal Sector 2022 V1.1,0.824567,
2,2,bao_viet_holdings_2023_p002_jpg.rf.6d94b20408c...,COMPATIBLES WITH DEVICES,text,0.556957,GRI 12,GRI 12_ Coal Sector 2022 V1.1,0.827402,
3,3,bao_viet_holdings_2023_p003_jpg.rf.a7d217e31e6...,WITH BAOVIET,text,0.826247,GRI 12,GRI 12_ Coal Sector 2022 V1.1,0.808983,
4,4,bao_viet_holdings_2023_p003_jpg.rf.a7d217e31e6...,"In 2023, the ""storm"" of the Covid-19 pandemic\...",text,0.996383,ghg-protocol-revised,ghg-protocol-revised,0.971892,


In [ ]:
rows = []

for item in semantic_blocks:

    match = item["top_matches"][0] if len(item["top_matches"]) else {}

    rows.append({

        "block_id": item["block_id"],

        "page": item["page"],

        "text": item["text"],

        "type": item["type"],

        "confidence": item["confidence"],

        "semantic_standard": item.get("semantic_standard"),

        "semantic_document": item.get("semantic_document"),

        "semantic_score": item.get("semantic_score"),

        "matched_chunk": match.get("text",""),

        "matched_page": match.get("page"),

        "matched_document": match.get("document"),

        "scope":""

    })

df = pd.DataFrame(rows)
print(df.shape)

df.head()

(13236, 12)


,block_id,page,text,type,confidence,semantic_standard,semantic_document,semantic_score,matched_chunk,matched_page,matched_document,scope
0,0,bao_viet_holdings_2023_p002_jpg.rf.6d94b20408c...,Ggo0 APPRECIATE WHAT WE HAVE\nENKIRONMENTAL wh...,figure,0.560416,GRI 2,GRI 2_ General Disclosures 2021,0.839976,; modified Note: See Guidance to Disclosure 2-...,52,GRI 2_ General Disclosures 2021,
1,1,bao_viet_holdings_2023_p002_jpg.rf.6d94b20408c...,60 9400\nSOCIAL Ve'll be shoring,figure,0.446182,GRI 12,GRI 12_ Coal Sector 2022 V1.1,0.824567,". 188. European Parliament, Committee on Forei...",79,GRI 12_ Coal Sector 2022 V1.1,
2,2,bao_viet_holdings_2023_p002_jpg.rf.6d94b20408c...,COMPATIBLES WITH DEVICES,text,0.556957,GRI 12,GRI 12_ Coal Sector 2022 V1.1,0.827402,". 188. European Parliament, Committee on Forei...",79,GRI 12_ Coal Sector 2022 V1.1,
3,3,bao_viet_holdings_2023_p003_jpg.rf.a7d217e31e6...,WITH BAOVIET,text,0.826247,GRI 12,GRI 12_ Coal Sector 2022 V1.1,0.808983,". 188. European Parliament, Committee on Forei...",79,GRI 12_ Coal Sector 2022 V1.1,
4,4,bao_viet_holdings_2023_p003_jpg.rf.a7d217e31e6...,"In 2023, the ""storm"" of the Covid-19 pandemic\...",text,0.996383,ghg-protocol-revised,ghg-protocol-revised,0.971892,. All too often team members left meet- ings w...,17,ghg-protocol-revised,


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df)

https://docs.google.com/spreadsheets/d/1WQUzSN-wY24mZxLwdYpt7lenldC2WiIHWwAv3FMzpAI/edit#gid=0
